In [ ]:
import pandas as pd
import numpy as np

In [ ]:
orders_df = pd.read_csv("data/ecommerce_dataset/orders.csv").sort_values("order_date")
products_df = pd.read_csv("data/processed/products_processed.csv").sort_values("updated_at")
order_items_df = pd.read_csv("data/ecommerce_dataset/order_items.csv")

products_df['updated_at'] = pd.to_datetime(products_df['updated_at'])
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

In [ ]:
orders_df = orders_df.reset_index(drop=True)
orders_df.head(2)

In [ ]:
products_df = products_df.reset_index(drop=True)
products_df.head(2)

In [ ]:
order_items_df = order_items_df.reset_index(drop=True)
order_items_df.head(2)

In [ ]:
order_items_expanded = order_items_df.merge(
    orders_df[['order_id', 'order_date']],
    on='order_id',
    how='left'
)

order_items_expanded.head(3)

In [ ]:
order_items_expanded = order_items_expanded.sort_values(
    ['order_date', 'order_id']
).reset_index(drop=True)

order_items_expanded.head(3)

In [ ]:
orders_items_processed = pd.merge_asof(
    order_items_expanded,
    products_df,
    left_on="order_date",
    right_on="updated_at",
    by="product_id",
    direction="backward"
)

orders_items_processed['item_price'] = orders_items_processed['price']
orders_items_processed['item_total'] = orders_items_processed['item_price'] * orders_items_processed['quantity']

orders_items_processed.sort_values(by='updated_at', ascending=False).head(3)

In [ ]:
orders_items_processed['item_price'] = orders_items_processed['price']
orders_items_processed['item_total'] = orders_items_processed['item_price'] * orders_items_processed['quantity']

orders_items_processed.sort_values(by='updated_at', ascending=False).head(3)

In [ ]:
orders_items_processed = orders_items_processed.drop(columns=['order_date', 'product_name',
                                                              'category', 'brand', 'price',
                                                              'rating', 'updated_at'])

orders_items_processed.head()

In [ ]:
order_total_amounts = orders_items_processed.groupby('order_id').agg(
    new_total_amount=('item_total', 'sum')
).reset_index()

order_total_amounts.head()

In [ ]:
orders_processed = orders_df.merge(
    order_total_amounts,
    how='left',
    on='order_id'
)

orders_processed.loc[
    ~np.isclose(orders_processed['total_amount'], orders_processed['new_total_amount'], atol=0.01),
    ['order_id', 'total_amount', 'new_total_amount']
]
#orders_processed.sort_values(by='order_date', ascending=False).head()

In [ ]:
orders_processed['total_amount'] = orders_processed['new_total_amount']
orders_processed = orders_processed.drop(columns='new_total_amount')

orders_processed.head()

## Validating the Source DBs seeding

In [2]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(os.getenv("POSTGRES_URL"))

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM source.orders"))
    print("Orders count:", result.scalar())

    result = conn.execute(text("SELECT * FROM source.orders LIMIT 5"))
    for row in result:
        print(row)

Orders count: 20000
('O00006843', 'U003174', datetime.datetime(2024, 1, 1, 0, 14, 17, 631198), 'shipped', 283.17)
('O00012483', 'U002516', datetime.datetime(2024, 1, 1, 0, 37, 29, 586479), 'processing', 21.52)
('O00015536', 'U003437', datetime.datetime(2024, 1, 1, 1, 20, 7, 693955), 'returned', 127.76)
('O00007716', 'U000721', datetime.datetime(2024, 1, 1, 1, 54, 47, 44827), 'returned', 1499.08)
('O00019145', 'U009774', datetime.datetime(2024, 1, 1, 2, 51, 9, 400801), 'cancelled', 1127.04)


In [3]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM source.order_items;"))
    print("Order Items count:", result.scalar())

    result = conn.execute(text("SELECT * FROM source.order_items LIMIT 5"))
    for row in result:
        print(row)

Order Items count: 43525
('I00014861', 'O00006843', 'P000474', 'U003174', 1, 283.17, 283.17)
('I00027148', 'O00012483', 'P000532', 'U002516', 1, 21.52, 21.52)
('I00033828', 'O00015536', 'P001436', 'U003437', 2, 63.88, 127.76)
('I00016701', 'O00007716', 'P000070', 'U000721', 1, 802.45, 802.45)
('I00016702', 'O00007716', 'P001437', 'U000721', 1, 47.35, 47.35)
